# Appendix A: Introduction to PyTorch (Part 1)

# A.1. What is PyTorch

In [49]:
import torch

print(torch.__version__)

2.13.0+cpu


In [50]:
torch.cuda.is_available()

False

| S.No | Library | Description |
| :--- | :--- | :--- |
|1| Tensor library | PyTorch implements a tensor (array) library for efficient computing |
|2| Automatic differentiation engine | PyTorch's Includes utilities to differentiate computations automatically |
|3|Deep Learning library| PyTorch's deep learning utilities make use of its tensor library and automatic differentiation engine|

```mermaid

```

# A.2. Understanding tensors

## A.2.1. Scalars, Vectors, Matrices, and Tensors

In [51]:
import torch
import numpy as np

# create a 0D tensor (scalar) from a Python integer
tensor0d = torch.tensor(1)
tensor0d

# create a 1D tensor (vector) from a PYthon list
tensor1d = torch.tensor([1,2,3])
tensor1d
# create a 2D tensor from a nested Python list
tensor2d = torch.tensor([[1,2],[3,4]])

# create a 3D tensor from a nested Python list
tensor3d_1 = torch.tensor([[[1,2],[3,4]],[[5,6],[7,8]]])

# create a 3D tensor from NumPy array
ary3d = np.array([[[1,2],[3,4]],[[5,6],[7,8]]])

tensor3d_2 = torch.tensor(ary3d)
tensor3d_2

tensor3d_3 = torch.from_numpy(ary3d)
tensor3d_3


tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [52]:
# Remains unchanged
ary3d[0,0,0] = 999
tensor3d_2

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [53]:
# Changes because of memory sharing
tensor3d_3

tensor([[[999,   2],
         [  3,   4]],

        [[  5,   6],
         [  7,   8]]])

## A.2.2. Tensor data types

In [54]:
tensor1d = torch.tensor([1,2,3])
tensor1d

tensor([1, 2, 3])

In [55]:
floatvec = torch.tensor([1.0, 2.0, 3.0])
floatvec.dtype

torch.float32

In [56]:
floatvec = tensor1d.to(torch.float32)
floatvec.dtype

torch.float32

## A.2.3. Common PyTorch tensor operations

In [57]:
tensor2d = torch.tensor([[1,2,3],[4,5,6]])
tensor2d

tensor([[1, 2, 3],
        [4, 5, 6]])

In [58]:
tensor2d.shape

torch.Size([2, 3])

In [59]:
tensor2d.reshape(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [60]:
tensor2d.view(3,2)

tensor([[1, 2],
        [3, 4],
        [5, 6]])

In [61]:
tensor2d.T

tensor([[1, 4],
        [2, 5],
        [3, 6]])

In [62]:
tensor2d.matmul(tensor2d.T)

tensor([[14, 32],
        [32, 77]])

In [63]:
tensor2d @ tensor2d.T

tensor([[14, 32],
        [32, 77]])

# A.3. Seeing models as computation graphs

In [64]:
import torch.nn.functional as F 

def T(v,grad=False):
    return torch.tensor(v,requires_grad=grad)

def S(v):
    return torch.sigmoid(v)

def E(a,o):
    return F.binary_cross_entropy(a,o)

y  = T([1.0])
x1 = T([1.1])
w1 = T([2.2])
b  = T([0.0])

z = x1 * w1 + b 
a = S(z)

loss = E(a,y)
loss

tensor(0.0852)

# A.4. Automatic differentiation made easy

In [65]:
import torch.nn.functional as F 
from torch.autograd import grad 


def P(loss, n, graph=True): 
  return grad(loss,n,retain_graph=graph)

y  = T([1.0])
x1 = T([1.1])
w1 = T([2.2],True)
b  = T([0.0],True)

z = x1 * w1 + b 
a = S(z)

loss = E(a,y)
loss

P_L_W1 = P(loss, w1)
P_L_b  = P(loss,b)

(P_L_W1, P_L_b)




((tensor([-0.0898]),), (tensor([-0.0817]),))

In [66]:
loss.backward()

(w1.grad, b.grad)

(tensor([-0.0898]), tensor([-0.0817]))

# A.5. Implementing multilayer neural networks

In [67]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, n_input, n_output):
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Linear(n_input, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30,20), 
            torch.nn.ReLU(),

            torch.nn.Linear(20, n_output)   
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits    

In [68]:
model = NeuralNetwork(50,3)
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [69]:
num_params = sum(p.numel() for p in model.parameters())
print("Total no. of trainable model parameters:", num_params)

Total no. of trainable model parameters: 2213


In [70]:
model.layers[0].weight

Parameter containing:
tensor([[ 0.1388,  0.0159,  0.1215,  ...,  0.1032,  0.0296,  0.0102],
        [ 0.0229,  0.0260, -0.0458,  ..., -0.0358,  0.0362,  0.0497],
        [-0.0896,  0.0113,  0.1370,  ...,  0.1037,  0.1230, -0.0929],
        ...,
        [-0.1362, -0.0713, -0.0010,  ...,  0.1176,  0.1054, -0.1012],
        [ 0.1226,  0.0937, -0.1409,  ...,  0.1321, -0.0613,  0.0086],
        [-0.0045, -0.0604,  0.0535,  ...,  0.0697,  0.0373,  0.0923]],
       requires_grad=True)

In [71]:
torch.manual_seed(123)
model = NeuralNetwork(50,3)
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

In [72]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [73]:
torch.manual_seed(123)

X = torch.rand((1,50))
out = model(X)
out

tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)

In [74]:
with torch.no_grad():
    out = model(X)
out

tensor([[-0.1262,  0.1080, -0.1792]])

In [75]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
out

tensor([[0.3113, 0.3934, 0.2952]])

# A.6. Setting up efficient data loaders

In [76]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [ 2.3,-1.1],
    [ 2.7,-1.5]
])

y_train = torch.tensor([0,0,0,1,1])

In [77]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [ 2.6,-1.6]
])

y_test = torch.tensor([0,1])

In [78]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self,X,y):
        self.features = X
        self.labels = y
    
    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y 
    
    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds  = ToyDataset(X_test, y_test)

        

In [79]:
len(train_ds)

5

In [80]:
from torch.utils.data import DataLoader

torch.manual_seed(123) 

def L(dataset,batch_size=2,shuffle=True,num_workers=0,drop_last=False):
    return DataLoader(dataset=dataset,batch_size=batch_size,shuffle=shuffle,num_workers=num_workers,drop_last=drop_last)



In [81]:
train_loader = L(train_ds)
test_loader  = L(test_ds,shuffle=False)

In [82]:
def batch(tl):
 for idx, (x,y) in enumerate(tl):
    print(f"Batch {idx+1}:",x,y)

batch(train_loader)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [83]:
train_loader = L(train_ds,drop_last=True)
batch(train_loader)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


# A.7. A Typical training loop

In [84]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(n_input=2,n_output=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss   = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ### Log
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx+1:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
    model.eval()
    

Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 002/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 002/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 002/002 | Train/Val Loss: 0.00


In [85]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [86]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])


In [87]:
predictions = torch.argmax(outputs, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [88]:
predictions == y_train

tensor([True, True, True, True, True])

In [89]:
torch.sum(predictions == y_train)

tensor(5)

In [90]:
def compute_accuracy(model, dataloader):

    model.eval()
    correct = 0.0
    total_examples = 0
    
    for idx, (features, labels) in enumerate(dataloader):
        
        with torch.no_grad():
            logits = model(features)
        
        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

In [91]:
compute_accuracy(model, train_loader)

1.0

In [92]:
compute_accuracy(model, test_loader)

1.0

# A.8. Saving and loading models

In [93]:
torch.save(model.state_dict(), "model.pth")

In [94]:
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [95]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


----------------------------------------------------------------------------------------------------------
# I not have the GPU Device, Can anyone offer GPU Supported Desktop or Workstation or Supercomputer or Quantum Computer
----------------------------------------------------------------------------------------------------------


# A.9. Optimizing training performance with GPUs

### A.9.1. PyTorch computations on GPU devices

### A.9.2. Single-GPU training

### A.9.3 Training with multiple GPUs